# RAG PIPELINE- ## data ingestion to vector db pipeline


In [52]:
import os
# os -> File aur folder paths manage karne ke liye

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
# PDF files se text extract karne ke liye
# Dono loaders alternatives hain; requirement ke according ek use karo

from langchain_text_splitters import RecursiveCharacterTextSplitter
# Large text ko smaller chunks mein divide karne ke liye (Chunking)

from pathlib import Path
# File aur folder paths ko easily handle karne ke liye

In [53]:
#Multiple PDFs ko automatically load karke ek common document list banana, jise hum next step mein chunking ke liye use karenge
def process_all_pdfs(pdf_directory):

    """Process all PDF files inside the directory"""

    all_documents = []  # Sabhi PDFs ke documents store karne ke liye

    pdf_dir = Path(pdf_directory)  # Folder path ko Path object mein convert kar rhe hain

    # Find all PDF files, including subfolders
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    # Total kitni PDF files mili hain
    print(f"Found {len(pdf_files)} PDF files to process")

    # Ek-ek PDF file ko process karenge
    for pdf_file in pdf_files:

        print(f"\nProcessing: {pdf_file.name}")

        try:
            # PDFLoader object create karke PDF load kar rhe hain
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Har document ke metadata mein source information add kar rhe hain
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            # Current PDF ke documents ko main list mein add kar rhe hain
            all_documents.extend(documents)

            print(f"✓ Loaded {len(documents)} pages")

        except Exception as e:
            # Agar kisi PDF ko load karne mein error aaye toh handle karenge
            print(f"✗ Error: {e}")

    # Total kitne documents/pages load hue
    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents  # Complete document list return kar rhe hain


# Process all PDFs inside data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: 01_python_programming.pdf
✓ Loaded 1 pages

Processing: 02_machine_learning.pdf
✓ Loaded 1 pages

Processing: 03_rag_and_langchain.pdf
✓ Loaded 1 pages

Processing: 04_cyber_security.pdf
✓ Loaded 1 pages

Total documents loaded: 4


In [54]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-31T06:36:29+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-31T06:36:29+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_python_programming.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_python_programming.pdf', 'file_type': 'pdf'}, page_content='Python Programming Introduction\nLearning notes for document-loading and RAG practice\nWhat is Python?\nPython is a high-level, interpreted programming language. It is known for readable syntax and a large\necosystem of libraries. Python was created by Guido van Rossum and first released in 1991.\nKey Features\nPython is easy to learn, supports multiple programming styles, runs on different operating systems, and\nhas a large standard library. It also has a strong developer community.\nCommon Data Types\nCommon bui

In [55]:


# Text splitting: Large documents ko smaller chunks mein divide karna
# Chunking RAG mein relevant information ko easily retrieve karne mein help karti hai.

def split_documents(documents, chunk_size=1000, chunk_overlap=200):

    # RecursiveCharacterTextSplitter object create kar rhe hain
    text_splitter = RecursiveCharacterTextSplitter(

        chunk_size=chunk_size,       
        # Har chunk mein maximum 1000 characters rakhne ki koshish karega.

        chunk_overlap=chunk_overlap, 
        # Consecutive chunks mein 200 characters common rahenge,
        # taaki previous aur next chunk ka context maintain rahe.

        length_function=len,         
        # Text ki length characters mein calculate karega.

        separators=["\n\n", "\n", " ", ""]
        # Split karne ki priority:
        # 1. \n\n -> Paragraph ke basis par
        # 2. \n   -> New line ke basis par
        # 3. " "  -> Words ke basis par
        # 4. ""   -> Individual characters ke basis par
    )

    # Original documents ko smaller chunks mein divide kar rhe hain
    split_docs = text_splitter.split_documents(documents)

    # Original documents aur generated chunks ka count show karega
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # First chunk ka example display kar rhe hain
    if split_docs:

        print("\nExample chunk:")

        # First chunk ka actual text (starting 200 characters)
        print(f"Content: {split_docs[0].page_content[:200]}...")

        # First chunk ki metadata (source file, page number etc.)
        print(f"Metadata: {split_docs[0].metadata}")

    # Saare generated chunks function ke bahar return kar rhe hain
    return split_docs


# All PDF documents ko chunks mein convert kar rhe hain
chunks = split_documents(all_pdf_documents)

# Total generated chunks count karna
len(chunks)

Split 4 documents into 8 chunks

Example chunk:
Content: Python Programming Introduction
Learning notes for document-loading and RAG practice
What is Python?
Python is a high-level, interpreted programming language. It is known for readable syntax and a lar...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-31T06:36:29+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-31T06:36:29+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_python_programming.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_python_programming.pdf', 'file_type': 'pdf'}


8

## Embedding and vector storedb 

in requirements.txt , i will install something like sentence-transformers,faiss-cpu,chromadb
1.Suppose humare paas do sentences hain:
- "I love programming."
- "I enjoy coding."
Dono sentences ka meaning similar hai.

Sentence Transformers inhe numerical vectors mein convert karega. Similar meaning wale sentences ke vectors generally ek doosre ke closer honge.
2.faiss -Jab humare paas thousands ya millions of embeddings hote hain, FAISS unmein se query ke most similar vectors quickly search karta hai.
3.Jab hum documents ke chunks ko embeddings mein convert kar lete hain, toh un embeddings ko store karne aur baad mein search karne ke liye ChromaDB use karte hain.


In [56]:
import numpy as np
# Numerical vectors ko handle karne ke liye

from sentence_transformers import SentenceTransformer
# Text ko numerical embeddings mein convert karne ke liye

import chromadb
# Embeddings aur documents ko vector database mein store/search karne ke liye

from chromadb.config import Settings
# ChromaDB ki configuration manage karne ke liye

import uuid
# Har document/chunk ko unique ID generate karne ke liye

from typing import List, Dict, Any, Tuple
# Variables aur function ke data types specify karne ke liye
# List = List of items, Dict = Key-value pairs
# Any = Any data type, Tuple = Fixed group of values

from sklearn.metrics.pairwise import cosine_similarity
# Do vectors ke beech similarity measure karne ke liye
# Similarity jitni high, meaning utna similar

niche wala code k liye 
Isko kyun bana rahe hain?
Taaki baar-baar embedding model load na karna pade. Ek class ke through model load aur embedding-related operations manage kar sakein.

Example: all-MiniLM-L6-v2 model 384-dimensional embeddings generate karta hai.
all-MiniLM-L6-v2 model text ko numerical vectors (embeddings) mein convert karta hai.

Ye Hugging Face par available pre-trained model hai, jise hum SentenceTransformer library se use karte hain.

In [57]:
class EmbeddingManager:
    # Document embeddings generate karne ke liye class

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        # Constructor: Object create hote hi automatically call hota hai

        self.model_name = model_name
        # Model ka naam store kar rhe hain

        self.model = None
        # Abhi model load nahi hua, isliye None rakha hai

        self._load_model()
        # Model ko load karne ke liye function call


    def _load_model(self):
        # Pre-trained SentenceTransformer model load karne ka function

        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)
            # Pre-trained model load kar rhe hain
            # Ye text ko numerical embeddings mein convert karega

            print(
                f"Model loaded successfully. Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )
            # Embedding vector ki dimensions check kar rhe hain

        except Exception as e:
            # Model load karte waqt error aaye toh handle karenge

            print(f"Error loading model {self.model_name}: {e}")

            raise
            # Error ko dobara raise karenge, taaki calling code ko pata chale


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        # texts -> Input mein strings ki list aayegi
        # np.ndarray -> Function NumPy array (numerical vectors) return karega

        """
        Generate embeddings for a list of texts.
        """

        # Check kar rhe hain ki embedding model load hua hai ya nahi
        if not self.model:
            raise ValueError("Model not loaded")
            # Model load nahi hai toh error generate karke execution stop karega

        # Total kitne texts ke embeddings generate karne hain, count show karega
        print(f"Generating embeddings for {len(texts)} texts...")

        # SentenceTransformer model text ko numerical vectors mein convert karega
        # show_progress_bar=True -> Embedding generation ka progress show karega
        embeddings = self.model.encode(texts, show_progress_bar=True)

        # Generated embeddings ka shape show karega
        # Shape = (Number of texts, Embedding dimensions)
        print(f"Generated embeddings with shape: {embeddings.shape}")

        # Generated numerical vectors ko function ke bahar return karega
        return embeddings


    # it is optional to see the dimension or the function
    def get_embedding_dimension(self) -> int:
        # self -> Current class object ko access karne ke liye
        # -> int -> Function integer value return karega

        # Model load hua hai ya nahi, check kar rhe hain
        if not self.model:
            raise ValueError("Model not loaded")
            # Model load nahi hai toh error generate karega

        # Model ke embedding vector ki dimension return karega
        # Example: all-MiniLM-L6-v2 -> 384 dimensions
        return self.model.get_sentence_embedding_dimension()


# Initialize the EmbeddingManager class
# Class ka object create kar rhe hain

embedding_manager = EmbeddingManager()
# __init__() automatically call hoga
# Default model "all-MiniLM-L6-v2" load hoga
# _load_model() execute hokar model initialize karega

# Object display kar rhe hain
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2846.26it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\RITIKA JAISWAL\AppData\Local\Temp\ipykernel_16356\3537556009.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


now, database is converted into vectors, so embedding is done, now we will store this vectors in vectordb

# vectorDb / vectorstore


In [58]:
class VectorStore:
    
    # ChromaDB mein embeddings aur documents store/retrieve karne ke liye class

    def __init__(
        self,
        collection_name: str = "pdf_documents_cosine",  # New collection name
        persist_directory: str = "./data/vector_store"
    ):
        # Object create hote hi database ki settings initialize hongi

        self.collection_name = collection_name
        # Collection ka naam save kar rhe hain

        self.persist_directory = persist_directory
        # Database ka permanent storage path save kar rhe hain

        self.client = None
        # Initially database connection nahi hai

        self.collection = None
        # Initially collection available nahi hai

        self._initialize_store()
        # Database connection aur collection initialize karenge


    def _initialize_store(self):
        # ChromaDB client aur collection initialize karne ka function

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            # Database ke liye folder create kar rhe hain

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )
            # ChromaDB se connection establish kar rhe hain

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine"  # Cosine distance metric set kar rhe hain
                }
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    # Method 2: Documents aur embeddings ko ChromaDB mein add karna
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        # Har document ka corresponding embedding hona chahiye
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        # Total kitne documents add honge, show karega
        print(f"Adding {len(documents)} documents to vector store...")

        # ChromaDB ke liye 4 lists prepare kar rhe hain
        ids = []             
        # Har document ki unique ID

        metadatas = []       
        # Source, page number aur additional information

        documents_text = []  
        # Documents ka original text

        embeddings_list = [] 
        # Documents ke numerical vectors


        # Har document ko uski corresponding embedding ke saath pair kar rhe hain
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # Har document ke liye unique ID generate kar rhe hain
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"

            # Unique ID ko list mein add kar rhe hain
            ids.append(doc_id)


            # Existing metadata ki copy bana rhe hain
            metadata = dict(doc.metadata)

            # Document ka index aur text length add kar rhe hain
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)

            # Prepared metadata ko list mein add kar rhe hain
            metadatas.append(metadata)


            # Document ka original text store kar rhe hain
            documents_text.append(doc.page_content)


            # NumPy embedding ko Python list mein convert kar rhe hain
            # ChromaDB ko embeddings list ke format mein de rhe hain
            embeddings_list.append(embedding.tolist())


        # Ab saara prepared data actual database mein insert karenge
        try:
            self.collection.add(
                ids=ids,                       # Unique IDs
                embeddings=embeddings_list,    # Numerical vectors
                metadatas=metadatas,           # Additional information
                documents=documents_text       # Original text
            )

            # Successfully add hue documents ka count show karega
            print(f"Successfully added {len(documents)} documents to vector store")

            # Collection mein ab total kitne records hain, show karega
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            # Data insert karte waqt error aaye toh handle karega
            print(f"Error adding documents to vector store: {e}")
            raise

 
        

Part 3: Ab tak Vector Database mein humne kya-kya kiya?
Isko apne notebook mein ek separate Markdown cell mein likh lena.
RAG – Vector Store Pipeline
1. Document Loading
   
   PDF files ko PyPDFLoader se load kiya.
   
   PDF → Document Objects
2. Text Extraction
   
   PDF se actual text aur metadata extract kiya.
   
   Page content + Source + Page number
3. Text Splitting / Chunking
   
   RecursiveCharacterTextSplitter se large text ko smaller chunks mein divide kiya.
   
   Chunk size = 1000, Overlap = 200
4. Embedding Model Initialization
   
   SentenceTransformer("all-MiniLM-L6-v2") model load kiya.
   
   Pre-trained Hugging Face model
5. Generate Embeddings
   
   model.encode() se har chunk ko 384-dimensional numerical vector mein convert kiya.
   
   Text → Numerical Vectors
6. VectorStore Initialization
   
   ChromaDB ka PersistentClient aur collection create/access ki.
   
   Database + Collection ready
7. Prepare Data
   
   Har chunk ke liye ID, metadata, original text aur embedding ki lists prepare ki.
   
   Data organized for insertion
8. Add Documents to Vector Store
   
   collection.add() se saara prepared data ChromaDB mein insert kiya.
   
   Documents + Embeddings + Metadata + IDs stored

In [59]:
# VectorStore class ka object create kar rhe hain
# Isse __init__() automatically call hoga
# Aur ChromaDB connection + collection initialize hogi

vectorstore = VectorStore()

# Object display kar rhe hain
vectorstore

Vector store initialized. Collection: pdf_documents_cosine
Existing documents in collection: 8


In [60]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-31T06:36:29+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-31T06:36:29+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_python_programming.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_python_programming.pdf', 'file_type': 'pdf'}, page_content='Python Programming Introduction\nLearning notes for document-loading and RAG practice\nWhat is Python?\nPython is a high-level, interpreted programming language. It is known for readable syntax and a large\necosystem of libraries. Python was created by Guido van Rossum and first released in 1991.\nKey Features\nPython is easy to learn, supports multiple programming styles, runs on different operating systems, and\nhas a large standard library. It also has a strong developer community.\nCommon Data Types\nCommon bui

In [61]:
# STEP 1: Chunks ka text extract kar rhe hain
texts = [doc.page_content for doc in chunks]

# STEP 2: Pehle define kiya hua method ab call kar rhe hain
embeddings = embedding_manager.generate_embeddings(texts)

# STEP 3: Generated embeddings aur chunks ko ChromaDB mein store kar rhe hain
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 8 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.15it/s]

Generated embeddings with shape: (8, 384)
Adding 8 documents to vector store...
Successfully added 8 documents to vector store
Total documents in collection: 16


# retreiver pipeline from vectorstore
Data ingestion ke through documents ko chunks aur embeddings mein convert karke Vector DB mein store kar liya.
Ab user query ko bhi embedding mein convert karenge.
Query embedding ki help se Vector DB mein similar/relevant chunks search karenge.
Retrieved chunks ko context ke roop mein use karenge.

In [62]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        # VectorStore object ko current class ke saath attach kar rhe hain
        # Isse hum stored documents aur embeddings access kar payenge

        self.embedding_manager = embedding_manager
        # EmbeddingManager object ko attach kar rhe hain
        # Isse user query ki embeddings generate kar payenge

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        # User ki query aur retrieval settings print kar rhe hain
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        # User ki query ko numerical vector mein convert kar rhe hain
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            # Query embedding ki help se Vector DB mein similar documents search kar rhe hain
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                # NumPy embedding ko Python list mein convert karke pass kar rhe hain

                n_results=top_k
                # Kitne top similar documents retrieve karne hain
            )

            # Process results
            # Retrieved documents ko store karne ke liye empty list
            retrieved_docs = []

            # Check kar rhe hain ki koi document retrieve hua hai ya nahi
            if results['documents'] and results['documents'][0]:

                # Retrieved documents, metadata, distances aur IDs extract kar rhe hain
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                # Har retrieved document aur uske details ko access kar rhe hain
                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):

                    # Distance ko similarity score mein convert kar rhe hain
                    # ChromaDB ke cosine distance ke liye score = 1 - distance
                    similarity_score = 1 - distance

                    # Minimum similarity threshold check kar rhe hain
                    if similarity_score >= score_threshold:

                        # Relevant document ki details dictionary mein store kar rhe hain
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                # Filtering ke baad kitne documents mile, count show kar rhe hain
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                # Agar koi document retrieve nahi hua
                print("No documents found")

            # Retrieved documents return kar rhe hain
            return retrieved_docs

        except Exception as e:
            # Retrieval ke time error aaye toh handle kar rhe hain
            print(f"Error during retrieval: {e}")

            # Error aane par empty list return kar rhe hain
            return []

Kya kar rahe hain?

Pehle se banaye hue VectorStore object ko RAGRetriever class ke andar store kar rahe hain, taaki baad mein usi VectorStore se relevant documents retrieve kar sakein.

In [63]:
rag_retreiver=RAGRetriever(vectorstore,embedding_manager)

In [64]:
rag_retreiver.retrieve("what is rag and langchain")

Retrieving documents for query: 'what is rag and langchain'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.26it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_f302423c_4',
  'content': "Retrieval-Augmented Generation (RAG)\nLearning notes for document-loading and RAG practice\nWhat is RAG?\nRetrieval-Augmented Generation, or RAG, is a technique that gives a language model relevant\ninformation from an external knowledge source before it generates an answer. The model uses the\nretrieved information as context.\nWhy use RAG?\nA language model may not know private documents or recently updated information. RAG can connect\nthe model to an external knowledge base without retraining the model every time the documents\nchange.\nMain Stages\nThe main stages are ingestion, retrieval, augmentation, and generation. During ingestion, documents\nare loaded, parsed, split into chunks, converted into embeddings, and stored. During retrieval, relevant\nchunks are searched for a user's question. During augmentation, the retrieved text is added to the\nprompt. During generation, the language model writes an answer.\nDocuments and Metadata",
  '

### Problem:
- Pehle ChromaDB default L2 (Euclidean) distance use kar raha tha.
- Hum `1 - distance` formula se similarity score calculate kar rahe the.
- L2 distance ke liye ye formula correct nahi tha.
- Negative similarity scores aa rahe the, isliye documents threshold condition mein reject ho rahe the.

### Changes:
- Collection initialization mein `"hnsw:space": "cosine"` add kiya.
- New collection `pdf_documents_cosine` create ki.
- Existing chunks aur embeddings ko new collection mein store kiya.

### Why?
- Cosine distance ke liye `similarity_score = 1 - distance` formula appropriate hai.
- Ab relevant documents ki similarity check karke retrieve kar sakte hain.

ignore niche k code wo bs upr ki glti check krne k liye the


In [65]:
# ChromaDB mein total kitne documents stored hain
print(vectorstore.collection.count())

16


In [66]:
# Query ko embedding mein convert kar rhe hain
query_embedding = embedding_manager.generate_embeddings(
    ["what is rag and langchain"]
)[0]

# ChromaDB se top 5 similar documents search kar rhe hain
results = vectorstore.collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

# Retrieved documents aur unki distances check kar rhe hain
print("Documents:", results['documents'])
print("Distances:", results['distances'])

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.63it/s]

Generated embeddings with shape: (1, 384)
Documents: [["Retrieval-Augmented Generation (RAG)\nLearning notes for document-loading and RAG practice\nWhat is RAG?\nRetrieval-Augmented Generation, or RAG, is a technique that gives a language model relevant\ninformation from an external knowledge source before it generates an answer. The model uses the\nretrieved information as context.\nWhy use RAG?\nA language model may not know private documents or recently updated information. RAG can connect\nthe model to an external knowledge base without retraining the model every time the documents\nchange.\nMain Stages\nThe main stages are ingestion, retrieval, augmentation, and generation. During ingestion, documents\nare loaded, parsed, split into chunks, converted into embeddings, and stored. During retrieval, relevant\nchunks are searched for a user's question. During augmentation, the retrieved text is added to the\nprompt. During generation, the language model writes an answer.\nDocuments an

In [67]:
# ChromaDB collection ka distance metric check kar rhe hain
print(vectorstore.collection.metadata)

{'description': 'PDF document embeddings for RAG', 'hnsw:space': 'cosine'}


till now, we have generated the content, based on query, query is converted into embeddings then that embedding gets searched in vectordb, similar searches we got, then content is made = query+Retrieved Context+prompt, now its tym to give it to the lllm .

ye phle hi kihe  h
"We have also persisted it in a local directory" means:

Humne documents aur unki embeddings ko ChromaDB mein store karne ke saath-saath apne computer ke local folder mein permanently save bhi kiya hai.

Tumhare code mein:
persist_directory = "./data/vector_store"



- ./ → Current working directory.
- data/vector_store → Jahan ChromaDB ka data save ho raha hai.
- PersistentClient → Data ko disk par persist karta hai.
Benefit: Kernel restart ya notebook close karne ke baad bhi database ka data available rahega. Har baar documents dobara add karne ki zarurat nahi hogi.

# Integration vectordb context pipeline with llm output

install in requiremnets.txt , python dotenv and langchin groq

In [76]:
# Groq ke ChatGroq model ko import kar rhe hain
from langchain_groq import ChatGroq

# Environment variables access karne ke liye
import os

# .env file se API key load karne ke liye
from dotenv import load_dotenv
load_dotenv()

# Groq API key set kar rhe hain
groq_api_key = os.getenv("GROQ_API_KEY")

# 1. Groq LLM initialize kar rhe hain
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)

# 2. Simple RAG function: Retrieve context + Generate response
def rag_simple(query, retriever, llm, top_k=3):

    # STEP 1: VectorDB se relevant documents retrieve kar rhe hain
    results = retriever.retrieve(query, top_k=top_k)

    # STEP 2: Retrieved documents ka content combine karke context bana rhe hain
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    # STEP 3: Check kar rhe hain ki relevant context mila ya nahi
    if not context:
        return "No relevant context found to answer the question."

    # STEP 4: Context aur query ko prompt mein combine karenge
    prompt = """Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    # Prompt mein actual context aur query insert karke LLM ko bhej rhe hain
    response = llm.invoke([
        prompt.format(context=context, query=query)
    ])

    # LLM ka generated answer return kar rhe hain
    return response.content

In [78]:
answer=rag_simple("what is cyber secuirty?",rag_retreiver,llm)
print(answer)

Retrieving documents for query: 'what is cyber secuirty?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 97.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Cyber security is the practice of protecting computers, networks, applications, and data from unauthorized access, attacks, damage, or theft.


In [79]:
# --- Enhanced RAG Pipeline Features ---
# Simple RAG ko enhance kar rhe hain, taaki answer ke saath
# sources, confidence score aur optionally context bhi mile

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):

    """
    RAG pipeline with extra features:
    - Answer, sources, confidence score return karega
    - Optionally full context bhi return karega
    """

    # STEP 1: Query ke basis par relevant documents retrieve kar rhe hain
    # top_k = kitne documents chahiye
    # min_score = minimum similarity score
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    # Agar koi relevant document nahi mila toh default response return karenge
    if not results:
        return {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    # STEP 2: Retrieved documents ka content combine karke context bana rhe hain
    context = "\n\n".join([doc['content'] for doc in results])

    # STEP 3: Retrieved documents ke sources aur details collect kar rhe hain
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] + '...'
    } for doc in results]

    # STEP 4: Sabse high similarity score ko confidence ke roop mein le rhe hain
    confidence = max([doc['similarity_score'] for doc in results])

    # STEP 5: Context aur query ko prompt mein combine kar rhe hain
    # LLM ko retrieved information ke basis par answer dene ko bol rhe hain
    prompt = """Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    # STEP 6: Formatted prompt Groq LLM ko bhej rhe hain
    response = llm.invoke(
        prompt.format(context=context, query=query)
    )

    # STEP 7: Answer, sources aur confidence ko dictionary mein store kar rhe hain
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    # Agar return_context=True hai toh complete context bhi output mein add karenge
    if return_context:
        output['context'] = context

    # Final output return kar rhe hain
    return output

In [81]:
# RAG PDF se related question pooch rhe hain
result = rag_advanced(
    "What is Retrieval-Augmented Generation (RAG)?",
    rag_retreiver,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

# Generated answer display kar rhe hain
print("Answer:", result['answer'])

# Answer ke supporting PDF sources display kar rhe hain
print("Sources:", result['sources'])

# Highest similarity score display kar rhe hain
print("Confidence:", result['confidence'])

# Retrieved context ke first 300 characters display kar rhe hain
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Retrieval-Augmented Generation (RAG)?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.08it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Retrieval‑Augmented Generation (RAG) is a technique that first retrieves relevant information from an external knowledge source (e.g., a document store) and then feeds that retrieved text into a language model as context, enabling the model to generate answers that incorporate up‑to‑date or private information without needing to be retrained.
Sources: [{'source': '03_rag_and_langchain.pdf', 'page': 0, 'score': 0.710486650466919, 'preview': 'Retrieval-Augmented Generation (RAG)\nLearning notes for document-loading and RAG practice\nWhat is RAG?\nRetrieval-Augment...'}, {'source': '03_rag_and_langchain.pdf', 'page': 0, 'score': 0.710486650466919, 'preview': 'Retrieval-Augmented Generation (RAG)\nLearning notes for document-loading and RAG practice\nWhat is RAG?\nRetrieval-Augment...'}, {'source': '03_rag_and_langchain.pdf', 'page': 0, 'score': 0.4163273572921753, 'preview': "chunks are searched for a user's question. During augmentation, the retrieved text is added to the\nprompt

In [82]:
# score =0.71 k mtlb 71% query ar fesult match ho gye

In [83]:
# Advanced RAG Pipeline:
# Basic RAG ke comparison mein extra features add kar rhe hain.

# 1. Streaming:
# Answer ko ek saath show karne ke bajaye gradually/token by token display karna.

# 2. Citations:
# Answer kis PDF aur kis page ke content se generate hua, uska reference dena.

# 3. Query History:
# User ke previous questions aur answers ko store karna,
# taaki baad mein unhe access kar sakein.

# 4. Summarization:
# Long generated answer ko short aur easy summary mein convert karna.

# 5. Confidence Score:
# Retrieved documents ka highest similarity score calculate karna,
# jisse pata chale ki retrieved content query se kitna similar hai.

# 6. Source Tracking:
# Retrieved documents ka source name, page number aur preview store karna.

# 7. Context Retrieval:
# Query ke according VectorDB se relevant documents retrieve karke
# unka content LLM ko provide karna.

# 8. LLM Generation:
# Retrieved context aur user question ki help se final answer generate karna.

from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:

    def __init__(self, retriever, llm):
        # Retriever aur LLM objects initialize kar rhe hain
        self.retriever = retriever
        self.llm = llm

        # Previous queries aur answers store karne ke liye history
        self.history = []

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False
    ) -> Dict[str, Any]:

        # STEP 1: VectorDB se relevant documents retrieve kar rhe hain
        results = self.retriever.retrieve(
            question,
            top_k=top_k,
            score_threshold=min_score
        )

        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""

        else:
            # STEP 2: Retrieved documents ka content combine karke context bana rhe hain
            context = "\n\n".join(
                [doc['content'] for doc in results]
            )

            # STEP 3: Documents ke source, page, score aur preview collect kar rhe hain
            sources = [
                {
                    'source': doc['metadata'].get(
                        'source_file',
                        doc['metadata'].get('source', 'unknown')
                    ),
                    'page': doc['metadata'].get('page', 'unknown'),
                    'score': doc['similarity_score'],
                    'preview': doc['content'][:120] + '...'
                }
                for doc in results
            ]

            # STEP 4: Context aur question ko prompt mein combine kar rhe hain
            prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {question}

Answer:"""

            # STEP 5: Streaming option
            if stream:
                # Answer ko gradually display karne ka simulation
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end="", flush=True)
                    time.sleep(0.05)
                print()

            # STEP 6: LLM se final answer generate kar rhe hain
            response = self.llm.invoke([prompt])
            answer = response.content

        # STEP 7: Answer ke saath source citations attach kar rhe hain
        citations = [
            f"[{i+1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]

        answer_with_citations = (
            answer + "\n\nCitations:\n" + "\n".join(citations)
            if citations else answer
        )

        # STEP 8: Optional summarization
        summary = None
        if summarize and answer:
            # LLM se answer ka 2-sentence summary generate karwa rhe hain
            summary_prompt = (
                f"Summarize the following answer in 2 sentences:\n{answer}"
            )
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # STEP 9: Highest similarity score ko confidence score ke roop mein le rhe hain
        confidence = max(
            [doc['similarity_score'] for doc in results],
            default=0.0
        )

        # STEP 10: Query aur answer ko history mein save kar rhe hain
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        # STEP 11: Final result dictionary return kar rhe hain
        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'confidence': confidence,
            'history': self.history
        }

In [84]:
# Advanced RAG object create kar rhe hain
adv_rag = AdvancedRAGPipeline(rag_retreiver, llm)

# User se question input le rhe hain
question = input("Ask your question: ")

# User ke question par Advanced RAG pipeline execute kar rhe hain
result = adv_rag.query(
    question,
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True
)

# Final answer aur additional details display kar rhe hain
print("\nAnswer:", result['answer'])
print("\nSummary:", result['summary'])
print("\nSources:", result['sources'])
print("\nConfidence Score:", result['confidence'])
print("\nQuery History:", result['history'])

Retrieving documents for query: 'what is cyber security and langchain'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 72.34it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.

Context:
Cyber Security Fundamentals
Learning notes for document-loading and RAG practice
What is Cyber Security?
Cyber security is the practice of protecting computers, networks, applications, and data from
unauthorized access, attacks, damage, or theft.
Com

mon Threats
Common cyber threats include malware, phishing, ransomware, password attacks, denial-of-service
attacks, and social engineering. Malware is harmful software. Ransomware may lock or encrypt files and
demand payment.
Phishing
Phishing is a social-engineering technique in which an attacker sends a deceptive message to trick
someone into revealing sensitive information or opening a harmful link or attachment. Users should
verify the sender and avoid unexpected links.
Basic Protection
Use strong and unique passwords, enable multi-factor authentication, install security updates, back up
important data, use trusted software, and be careful with links and attachments.
Encryption

Cyber Security Fundamentals
Learning notes for document-loading and RAG practice
What is Cyber Security?
Cyber security is the practice of protecting computers, networks, applications, and data from
unauthorized access, attacks, damage, or theft.
Common Threats
Common cyber threats include malware, phishin

ye upr m search jaha krte waha ques likhna tha hmko ar enter press krte hi ans aagya yaha bdhiya sa 

shprt m , ye v likhna src m ye ye bnaybgeg















Modular RAG Pipeline
Ab hum apne existing RAG pipeline ko modular structure mein divide karenge, taaki code clean, reusable aur easy to maintain rahe.
1. src folder ke andar ye files banayenge
File	Work
data_loader.py	PDF aur text files load karna
embedding.py	Text ko numerical embeddings mein convert karna
vectorstore.py	ChromaDB mein documents aur embeddings store karna
search.py	Query ke according relevant documents retrieve karna

2. Working order
1. data_loader.py → Documents load karenge.
2. embedding.py → Documents ke embeddings generate karenge.
3. vectorstore.py → Embeddings aur documents VectorDB mein store karenge.
4. search.py → User query se relevant documents search karenge.
3. Next step
In sabhi modules ko ek main file (main.py) mein import karke connect karenge.

Goal: Same RAG pipeline ko modular way mein implement karna, jahan har file ka ek specific responsibility hoga.